**보행 지원 시스템 2차 개발 — 모델 비교 → 라벨링 → 학습 → 성능 검증**

기존 5개 모델과 탐색 결과를 보존하고 9종 클래스로 B0/B2를 학습하는 연구용 노트북입니다.
실제 학습 데이터와 검수 라벨은 아직 없습니다. 데이터가 없는 단계는 실행하지 않으며 성능을 만들지 않습니다.
점자블록·계단 학습, 깊이·미터 거리 추정은 이번 범위에 포함하지 않습니다.
합성 테스트는 실제 학습·평가와 분리해 저장합니다. GPU·Drive 설정 후 필요한 구역을 순서대로 실행하세요.


# 1. 기존 5개 모델 분할 코드
설치·환경·Drive 설정과 기존 추론 기능입니다. 모델마다 새 Python 프로세스에서 실행합니다.


In [ ]:
# 1-1. GitHub 코드 가져오기와 라이브러리 설치
import sys, subprocess, json, importlib.metadata as metadata
from pathlib import Path

REPO_URL = 'https://github.com/i3l3/sidewalk-segmentation-benchmark.git'
REPO_REF = 'main'  # 재현 실험은 여기에 기록된 커밋 SHA를 넣어 고정하세요.
REPO_DIR = Path('/content/sidewalk-segmentation-benchmark')

def git(*args):
    return subprocess.check_output(['git', '-C', str(REPO_DIR), *args], text=True).strip()

if not (REPO_DIR / '.git').exists():
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        raise RuntimeError('코드 폴더가 비어 있지 않습니다. REPO_DIR에 다른 경로를 지정하세요.')
    subprocess.run(['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO_DIR)], check=True)
if git('remote', 'get-url', 'origin') != REPO_URL:
    raise RuntimeError('다른 저장소입니다. REPO_DIR에 별도 경로를 지정하세요.')
if git('status', '--porcelain'):
    raise RuntimeError('코드 폴더에 수정 사항이 있습니다. 변경 내용을 보존하거나 별도 경로를 사용하세요.')
git('fetch', '--depth', '1', 'origin', REPO_REF)
REPO_COMMIT = git('rev-parse', 'FETCH_HEAD')
git('checkout', '--detach', REPO_COMMIT)
print('사용 코드 커밋:', REPO_COMMIT)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements.txt')])
import torch
print('Python', sys.version, 'Torch', torch.__version__, 'Transformers', metadata.version('transformers'))
print('GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음: 실제 학습에는 GPU 런타임 필요')
assert tuple(map(int, torch.__version__.split('+')[0].split('.')[:2])) >= (2, 6)


## 선택 설정: Tailscale 국내 exit node 연결
바로 위 Git clone·설치 셀을 먼저 실행한다. 아래 `RUN_TAILSCALE=True`로 설치·로그인 셀을 실행한 뒤, 출력된 링크에서 **기존 국내 exit node와 같은 tailnet**에 로그인한다. 관리자 기기 승인이 필요한 tailnet은 Colab 기기도 승인한다. 다음 셀에서 사용 가능한 exit node를 조회하고 `TAILSCALE_EXIT_NODE`에 이름 또는 Tailscale IP를 넣어 다시 실행한다.

Tailscale은 userspace 모드의 로컬 HTTP 프록시로 실행한다. AI Hub 파일 목록·도구 설치·다운로드만 이 프록시를 사용하며 전역 네트워크 경로는 변경하지 않는다. 로그인 상태는 RAM에만 저장하고 Drive에 저장하지 않는다. 런타임이 새로 만들어지면 다시 로그인해야 한다. 연결 셀의 출력 IP가 국내 exit node의 공인 IP인지 확인한다. 연결은 AI Hub의 인증·승인·접근 허용을 보장하지 않는다.

공식 문서: [userspace networking](https://tailscale.com/docs/concepts/userspace-networking), [exit nodes](https://tailscale.com/docs/features/exit-nodes).

In [ ]:
# 선택 1: Tailscale 설치·로그인 (인증 키를 코드에 넣을 필요 없음)
RUN_TAILSCALE = False
TAILSCALE_EXIT_NODE = ''  # 다음 셀 목록에서 국내 exit node 이름 또는 100.x IP 입력
if RUN_TAILSCALE:
    import sys, importlib
    from pathlib import Path
    _src = sys.modules.get('src')
    if _src is not None and Path(_src.__file__).resolve().parent != (REPO_DIR / 'src').resolve():
        for _name in list(sys.modules):
            if _name == 'src' or _name.startswith('src.'):
                del sys.modules[_name]
    sys.path.insert(0, str(REPO_DIR))
    importlib.invalidate_caches()
    from src import tailscale_proxy
    tailscale_proxy.start_login()
else:
    print('Tailscale을 사용하려면 RUN_TAILSCALE=True로 바꾸고 실행하세요.')

In [ ]:
# 선택 2: 브라우저 로그인 후 실행. 목록에서 국내 exit node를 선택한다.
if RUN_TAILSCALE:
    from IPython.display import display
    import pandas as pd
    display(pd.DataFrame(tailscale_proxy.exit_nodes()))
    if TAILSCALE_EXIT_NODE:
        tailscale_proxy.select_exit_node(TAILSCALE_EXIT_NODE)
    else:
        print('위 목록에서 국내 exit node를 골라 TAILSCALE_EXIT_NODE에 입력한 뒤 다시 실행하세요.')

In [ ]:
# 선택 3: 다운로드를 마친 뒤 연결 해제가 필요할 때만 True로 바꿔 실행
DISCONNECT_TAILSCALE = False
if DISCONNECT_TAILSCALE:
    from src import tailscale_proxy
    tailscale_proxy.disconnect()

In [ ]:
from pathlib import Path
USE_DRIVE=True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Sidewalk-Segmentation-Benchmark') if USE_DRIVE else Path('/content/Sidewalk-Phase2')
ROOT.mkdir(parents=True,exist_ok=True)
BUNDLE=REPO_DIR  # 코드는 /content, 데이터·결과는 Drive ROOT에 저장
SOURCE_IMAGES=''        # 다운로드한 AI Hub 이미지 폴더 또는 ZIP 경로
REVIEWED_MASKS=''       # 외부 도구에서 수정한 ID PNG 폴더
REVIEWER=''             # 실제 검수자
CONFIRM_GROUPS=False    # dataset_metadata.csv의 group_id를 확인한 뒤 True
CONFIRM_REVIEWED=False  # 직접 라벨 검수를 완료한 경우에만 True
RUN_REAL_TRAINING=False # 자료와 검수 라벨이 준비된 후 True
EXPERIMENTS=['A','B','C','D']
BASELINE_MODELS=['mask2former_swin_l','oneformer_dinat_l','mask2former_r50','segformer_b5','segformer_b2']


In [ ]:
# 1-3. 데이터 경로와 독립 프로세스 실행 함수
import os, json, importlib
# A kernel may have imported src from an older checkout earlier in this session.
_cached_src = sys.modules.get('src')
if _cached_src is not None and Path(_cached_src.__file__).resolve().parent != (BUNDLE / 'src').resolve():
    for _module_name in list(sys.modules):
        if _module_name == 'src' or _module_name.startswith('src.'):
            del sys.modules[_module_name]
sys.path.insert(0, str(BUNDLE))
importlib.invalidate_caches()
from src.schema import init, save_json
init(ROOT)
(ROOT / 'data/excluded_hashes.json').write_bytes((BUNDLE / 'excluded_hashes.json').read_bytes())
save_json(ROOT / 'metrics/source_revision.json', {
    'repository': REPO_URL, 'requested_ref': REPO_REF, 'commit': REPO_COMMIT
})

def run(command, *args, root=ROOT):
    env = os.environ.copy()
    env['PYTHONPATH'] = str(BUNDLE)
    subprocess.run([
        sys.executable, '-u', '-m', 'src.runner', command,
        '--root', str(root), '--legacy', str(BUNDLE / 'legacy'), *map(str, args)
    ], cwd=str(BUNDLE), env=env, check=True)

print('GitHub 코드:', BUNDLE)
print('데이터·학습 결과:', ROOT)


### 기존 5개 모델 실행
탐색 사진은 `ROOT/exploration/inputs/korea` 또는 `other`에 둡니다. 기존 4장은 학습용 데이터 가져오기에서 해시로 제외됩니다.
기존 1차 노트북과 실험 결과는 이전 배포 자료에 보존되며, 기존 모델 로더는 저장소의 legacy 폴더에서 재사용합니다. 다운로드 가중치는 다시 묶어 배포하지 않습니다.


In [ ]:
EXPLORATION=ROOT/'exploration'
for key in BASELINE_MODELS:
    run('benchmark','--models',key,root=EXPLORATION)


# 2. 데이터셋 라벨링 코드
AI Hub 이미지 가져오기 → 그룹 분할 → 두 Teacher 초안 → 외부 검수 → Teacher 선택 순서입니다.


### 데이터 출처와 분할
[AI Hub 인도보행 영상](https://www.aihub.or.kr/aihubdata/data/view.do?aihubDataSe=data&currMenu=1&dataSetSn=189&topMenu=)을 우선 사용합니다.
계정·다운로드 인증과 이용 조건 확인은 사용자가 진행합니다. ZIP 또는 폴더를 준비한 뒤 SOURCE_IMAGES를 지정하세요.
목표는 train 250 / validation 50 / test 50이며, 실제 수량과 장면 그룹에 따라 조정됩니다.
`group_id`는 같은 장소·연속 프레임을 같은 그룹으로, `camera_view=pedestrian`은 확인한 사진에만 입력합니다.
그룹이 없으면 분할하지 않습니다. 테스트 목록 고정 후에는 새 데이터 버전/폴더로만 다시 나눕니다.


## AI Hub 공식 도구로 선택 다운로드
[인도보행 영상 데이터](https://www.aihub.or.kr/aihubdata/data/view.do?aihubDataSe=data&currMenu=1&dataSetSn=189&topMenu=)에서 **다운로드 이용 신청·승인**을 먼저 완료하고, [공식 aihubshell 안내](https://aihub.or.kr/devsport/apishell/list.do)에 따라 API 키를 발급받는다. Colab 왼쪽 열쇠(Secrets)에 `AIHUB_API_KEY`로 저장하고 이 노트북의 접근을 허용하거나, 실행 시 숨김 입력을 사용한다. 키를 코드에 적지 않는다.

아래 목록 조회 셀을 실행해 **원천 이미지 ZIP과 대응 라벨 ZIP**의 파일 키를 고른다. 영상 대신 보행자 시점 이미지를 선택하고 장소·시퀀스별 폴더를 보존한다. 데이터셋 전체 다운로드는 비활성화한다. 목록 조회는 공개 API 서비스 상태에 따라 실패할 수 있으며 실패하면 나중에 다시 시도한다. 다운로드 공간은 선택 용량의 3배를 요구한다. **공식 도구는 전송 TAR와 분할 파일을 병합하지만 내부 ZIP은 남으므로**, 원천 ZIP 경로를 다음 가져오기 셀의 `SOURCE_IMAGES`에 지정한다. 공식 라벨은 아직 검수 완료가 아니며, 실제 라벨 형식을 확인해 9종 ID로 변환해야 한다.

Colab 실조회에서 데이터셋 189의 54개 ZIP을 확인했다. 이번 작업에는 `서피스마스킹/Surface_*.zip`을 먼저 검토한다. 예: `Surface_1.zip`의 파일 키는 `49954`, 표기 용량 약 10 GB다. 목록은 변경될 수 있으므로 실행 시 다시 확인한다. ZIP 안에 원천 이미지와 라벨이 함께 있다면 **이미지 폴더만** 가져오고 라벨 폴더를 입력 이미지에 섞지 않는다.

### PC에서 직접 다운로드한 Surface ZIP
원본 ZIP에 JPG와 `MASK/*.png`, 폴더별 XML이 함께 들어 있다. `scripts/prepare_surface_subset.py`로 350쌍과 해당 XML만 보존한 작은 ZIP을 만든 뒤, Drive의 `Sidewalk-Segmentation-Benchmark/data/uploads/sidewalk_subset_350.zip`에 올린다. `SOURCE_IMAGES`는 `/content/drive/MyDrive/Sidewalk-Segmentation-Benchmark/data/uploads/sidewalk_subset_350.zip`으로 설정한다. 최신 코드의 이미지 가져오기는 MASK 폴더를 입력 사진에서 제외한다. 실제 PNG ID/색상 정의와 XML을 확인하기 전 학습 라벨로 사용하지 않는다. 폴더 이름을 장소 그룹으로 자동 확정하지 않는다.

In [ ]:
import importlib
from src import aihub_download as _aihub_download
importlib.reload(_aihub_download)
from src.aihub_download import list_files, selection, download_files
import pandas as pd
AIHUB_DATASET_KEY = '189'  # 목록이 다르면 AI Hub 공식 datasetkey 확인
AIHUB_FILE_KEYS = []      # 목록의 file_key 문자열을 명시적으로 입력
AIHUB_CACHE = ROOT / 'data/downloads/aihub_tool'
# 원본 데이터셋 저장 위치: True=Google Drive, False=Colab 임시 디스크
SAVE_DATASET_TO_DRIVE = False
if SAVE_DATASET_TO_DRIVE:
    if not Path('/content/drive/MyDrive').is_dir():
        from google.colab import drive
        drive.mount('/content/drive')
    AIHUB_DESTINATION = Path('/content/drive/MyDrive/Sidewalk-Segmentation-Benchmark/data/downloads/aihub_189')
else:
    AIHUB_DESTINATION = Path('/content/aihub_downloads')
print('원본 데이터셋 저장 위치:', AIHUB_DESTINATION)
# 설정 변경은 이후 시작하는 다운로드에 적용된다. 기존 파일은 자동 이동하지 않는다.
RUN_AIHUB_LIST = False
RUN_AIHUB_DOWNLOAD = False
aihub_catalog = None
if RUN_AIHUB_LIST or RUN_AIHUB_DOWNLOAD:
    aihub_catalog = list_files(AIHUB_CACHE, AIHUB_DATASET_KEY)
    display(pd.DataFrame(aihub_catalog))
    surface_files = [r for r in aihub_catalog if "Surface_" in r["name"]]
    if surface_files:
        print("이번 프로젝트에서 우선 검토할 서피스마스킹 ZIP:")
        display(pd.DataFrame(surface_files))
else:
    print('공식 다운로드: RUN_AIHUB_LIST=True로 파일 목록부터 조회하세요.')

In [ ]:
if RUN_AIHUB_DOWNLOAD:
    from getpass import getpass
    chosen = selection(aihub_catalog, AIHUB_FILE_KEYS)
    display(pd.DataFrame(chosen))
    print('선택 ZIP 용량(GiB):', round(sum(r['size_bytes'] for r in chosen) / 2**30, 3))
    _aihub_key = ''
    try:
        try:
            from google.colab import userdata
            _aihub_key = userdata.get('AIHUB_API_KEY')
        except Exception:
            _aihub_key = getpass('AI Hub API key (숨김 입력): ')
        downloaded = download_files(AIHUB_CACHE, AIHUB_DESTINATION, AIHUB_FILE_KEYS,
                                    _aihub_key, AIHUB_DATASET_KEY)
        print('다운로드 완료. 원천 이미지 ZIP 하나를 SOURCE_IMAGES에 지정하세요:')
        for archive in downloaded['archives']: print(archive)
    finally:
        _aihub_key = None
else:
    print('파일 키를 고른 뒤 RUN_AIHUB_DOWNLOAD=True로 실행하세요.')

### Surface XML에서 학습용 마스크 준비
`Surface_1.zip`은 JPG·RGB 마스크·CVAT XML을 함께 포함한다. 아래 셀은 폴더별로 분산해 350장을 선별하고 XML 폴리곤의 z_order에 따라 ID 마스크를 만든다. 인도=2, 차도(normal)=1, 자전거도로=8만 변환한다. 골목·점자블록·주의구역·횡단보도 속성·미표기는 255로 남긴다. 원본 RGB·XML·원래 클래스 마스크·속성은 따로 보존한다. 지원하지 않는 도형이 있는 이미지는 제외한다.

이 표본은 지원 클래스가 있는 이미지에서 선별한 자료로, 원본 전체를 대표하는 무작위 테스트 세트가 아니다. 9종 중 3종만 제공되므로 나머지 클래스의 학습·평가에는 추가 라벨링이 필요하다. `data/review/surface_review.csv`와 오버레이를 사람이 확인해야 한다. `group_hint`는 실제 장소·시퀀스 확정값이 아니며 검수·분할·학습을 자동 실행하지 않는다. 중단 후 동일 설정으로 다시 실행하면 저장된 샘플을 검증하고 이어간다. 원본 Surface ZIP에는 일반 이미지 가져오기 셀을 실행하지 않는다.

In [ ]:
# AI Hub Surface 원본 ZIP: 전체 압축 해제 없이 XML 기반 표본 준비
RUN_SURFACE_PREPARATION = False
SURFACE_SAMPLE_COUNT = 350
if RUN_SURFACE_PREPARATION:
    import importlib
    from src import surface_import
    importlib.reload(surface_import)
    surface_summary = surface_import.prepare_surface(ROOT, SOURCE_IMAGES, count=SURFACE_SAMPLE_COUNT, seed=42)
    print(surface_summary)
else:
    print('Surface ZIP은 위 SOURCE_IMAGES를 설정한 뒤 RUN_SURFACE_PREPARATION=True로 준비하세요.')

In [ ]:
if SOURCE_IMAGES and Path(SOURCE_IMAGES).name.lower().startswith('surface_') and Path(SOURCE_IMAGES).suffix.lower()=='.zip':
    print('Surface 원본 ZIP은 XML 표본 준비 셀을 사용하세요. 전체 이미지 가져오기를 건너뜁니다.')
elif SOURCE_IMAGES:
    run('import','--source',SOURCE_IMAGES)
else:
    print('아직 학습 이미지 없음: AI Hub 자료를 받은 뒤 SOURCE_IMAGES를 지정하세요.')
import pandas as pd
metadata_path=ROOT/'data/dataset_metadata.csv'
if metadata_path.exists():
    DATA=pd.read_csv(metadata_path,keep_default_na=False);display(DATA);DATA.info()
    print('group_id/camera_view/source/license를 편집할 파일:',metadata_path)


In [ ]:
if CONFIRM_GROUPS:run('split','--confirm-groups')
else:print('장소/시퀀스 group_id를 확인한 뒤 CONFIRM_GROUPS=True로 실행하세요.')


### 9개 클래스·초안 생성
0 Other, 1 Road, 2 Sidewalk, 3 Curb, 4 Crosswalk (plain), 5 Pedestrian Area,
6 Curb Cut, 7 Crosswalk Marking, 8 Bike Lane. 학습 제외 픽셀=255.
기존 12종의 점자블록(5)·계단(6)·주의구역(11)·ignore(65535)는 새 체계의 ignore로 변환합니다.
두 Teacher의 원본 ID 마스크는 별도 보존하고, 예측 불일치 영역을 검수 우선순위로 표시합니다. 일치=정답은 아닙니다.


In [ ]:
if (ROOT/'data/splits/locked.json').exists():run('teachers')
else:print('이미지와 그룹 분할이 먼저 필요합니다.')


### 외부 검수용 내보내기·가져오기
Teacher별 이미지/ID 마스크/오버레이/class 정의를 ZIP으로 내보냅니다.
외부 도구에서 원본 크기의 단일 채널 PNG(ID 0~8, ignore=255)로 수정하고 파일명을 image_id.png로 유지하세요.
RGB 마스크는 자동으로 추측하지 않습니다. 제공 API의 color_map에 RGB→ID를 명시해야 합니다.
공식 AI Hub 폴리곤도 `import_aihub_polygons`로 가져올 수 있으며, 공식 스키마와 속성을 확인해 명시적으로 매핑합니다.
검증·테스트 세트는 전부 사람이 검수해야 합니다. 검수자 입력과 확인 없이 reviewed로 바뀌지 않습니다.


In [ ]:
if (ROOT/'data/splits/locked.json').exists():
    run('review-export','--source',ROOT/'data/review/review_bundle.zip')
    print('외부 수정 후 가져오기:',ROOT/'data/review/review_bundle.zip')
if REVIEWED_MASKS and CONFIRM_REVIEWED:
    run('review-import','--source',REVIEWED_MASKS,'--reviewer',REVIEWER,'--confirm-reviewed')


In [ ]:
# 검수된 train/validation과 양쪽 Teacher 예측이 있을 때만 선택합니다. test는 사용하지 않습니다.
SELECT_TEACHER=False
if SELECT_TEACHER:run('teacher-select')


# 3. 모델 학습 또는 파인튜닝 코드
ImageNet 인코더 + 새 9종 분할 헤드. 실제 학습과 합성 코드 검증을 구분합니다.


### 학습 조건
A=B0 검수 라벨 / B=B0 검수+별도 자동 라벨 / C=B2 검수 / D=B2 검수+별도 자동 라벨.
공식 nvidia/mit-b0와 mit-b2 인코더의 고정 리비전을 기록하며, 새 분할 헤드의 초기 상태는 과거 학습 B0가 아닙니다.
seed42, 512×512, AdamW lr6e-5, weight_decay .01, 유효 배치8, 최대20epoch, patience5.
검수된 train과 validation이 없으면 실제 학습을 거부합니다. B/D에는 추가 자동 라벨 자료가 필요합니다.
각 epoch 종료 때 optimizer/scheduler/AMP/RNG/모델을 저장하고 동일 데이터·설정에서 재개합니다.
입력 이미지·마스크의 동일 기하 증강, 마스크 nearest-neighbor 보간을 적용합니다. OOM 시 microbatch를 줄여 새 실행 설정으로 기록하세요.


In [ ]:
TRAIN_CONFIG={'seed':42,'size':512,'epochs':20,'lr':6e-5,'weight_decay':.01,'effective_batch':8,'patience':5}
if RUN_REAL_TRAINING:
    for experiment in EXPERIMENTS:run('train','--experiment',experiment,'--config',json.dumps(TRAIN_CONFIG))
else:print('실제 데이터 학습 미실행: 이미지와 검수 라벨 확보 후 RUN_REAL_TRAINING=True')


### 합성 자료 학습·저장·재개 검증
별도 synthetic 경로에서 아주 작은 SegFormer를 1epoch 실행→저장→재개→2epoch→평가합니다.
중단 없이 2epoch 실행한 모델과 가중치를 비교합니다. 이 기록은 실제 B0/B2 파인튜닝 성능이 아닙니다.


In [ ]:
RUN_SYNTHETIC_SMOKE=False
if RUN_SYNTHETIC_SMOKE:
    import datetime
    synthetic=ROOT/'synthetic_validation'/datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    run('smoke',root=synthetic)


# 4. 성능 검증 코드
고정 테스트 정량 평가·오류 시각화·보행 영역·방향 후보·수행평가 보고서입니다.


### 같은 고정 테스트 세트의 모델 비교
기존 5개 모델과 A~D의 best checkpoint를 평가합니다. 모든 테스트 정답이 검수되어야 실행합니다.
미지원 개별 클래스는 N/A, 공통 Other/Road/Sidewalk mIoU는 별도 제공합니다.
IoU·Precision·Recall·Pixel Accuracy·Boundary F1(2px)·추론 시간·GPU 메모리를 저장합니다.
한국/비한국 보행자 그룹을 나눠 둘 다 있을 때만 `(2×비한국 공통 mIoU+한국 공통 mIoU)/3` 보조 점수를 계산합니다.
원본·GT·예측·오류 영역과 GT 기준 연석 확대 영역을 동일하게 저장합니다.


In [ ]:
RUN_FINAL_EVALUATION=False # validation에서 설정/체크포인트 선택을 마친 뒤에만 True
if RUN_FINAL_EVALUATION:
    for key in BASELINE_MODELS:run('baselines','--models',key)
    for experiment in EXPERIMENTS:
        if (ROOT/'checkpoints'/experiment/'best').exists():run('evaluate','--experiment',experiment)


### 보행 영역과 방향 특징
하단 60% 영역에서 인도/보행자 구역만 후보로 사용하고, 하단 5%·중앙 20%와 연결된 성분을 찾습니다.
각 좌/중/우 면적 비율, 차지하는 행 비율, 행별 최장 연속 폭의 중앙값/영상폭, 중심 편차, 연석 근접도를 계산합니다.
미터 거리나 안전 경로로 해석하지 않습니다. 차도·횡단보도·연석·자전거도로는 통행 후보에서 제외합니다.
신뢰할 별도 장애물 마스크가 없으면 obstacle_ratio는 결측값입니다. 연결 성분이 없으면 No clear path입니다.
원본을 보고 직접 방향을 입력하세요. 규칙 예측을 정답으로 복사하지 않습니다.


In [ ]:
from src.direction import set_direction_label
# 예: 직접 이미지를 검토한 후에만 아래 호출을 사용하세요.
# set_direction_label(ROOT,'im_실제ID','Straight',reviewer='검수자',confirmed=True)
DIRECTION_MODELS=['A'] # validation에서 선택한 Student. 다른 Student별 분포도 별도로 비교 가능
RUN_DIRECTION=False
if RUN_DIRECTION:
    for experiment in DIRECTION_MODELS:
        run('student-features','--experiment',experiment)
        run('direction','--models',experiment)


### 방향 분류와 결과 파일
분류기 입력은 direction_features의 실제 수치, 정답은 사람이 검수한 Left/Straight/Right/No clear path입니다.
각 방향별 train20·validation5·test5 이상일 때 Decision Tree와 Random Forest를 학습합니다.
부족하면 규칙 Baseline과 부족 사유만 기록합니다. 규칙 임계값/분류기 선택은 validation만 사용합니다.
`dataset_metadata.csv`, `segmentation_metrics.csv`, 모델별 `direction_features.csv`, PNG 그래프와 보고서를 생성합니다.
느낀 점과 최종 결론은 실제 관찰 후 본인이 작성합니다.


In [ ]:
run('report')
print((ROOT/'reports/experiment_report.md').read_text(encoding='utf-8'))
